# Aggregations

## What is it?
Grouping brings equal keys together, so it often shuffles data. Aggregation collapses many rows into one row per key.

## Why does it exist?
Spark lets a single program express parallel, fault-tolerant data work. Its optimizer can inspect DataFrame/SQL expressions before running them.

## When should I use it?
Use this technique when the data relationship or operational concern in the examples matches your problem; validate assumptions with data profiles and plans.

## How does it work?
```mermaid
flowchart LR
  A[DataFrame transformations] --> B[Logical plan]
  B --> C[Catalyst / physical plan]
  C --> D[Tasks over partitions]
  D --> E[Action result or files]
```

Compare grouped revenue and global totals; inspect why `groupBy` differs from a window calculation.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark_learning.spark_session import create_spark_session

spark = create_spark_session("Learning Lab: Aggregations")
raw = Path.cwd().parent / "data" / "raw"
if not raw.exists():
    raw = Path("data/raw")
orders = spark.read.option("header", True).csv(str(raw / "orders.csv"))
customers = spark.read.option("header", True).csv(str(raw / "customers.csv"))
orders.show(5, truncate=False)
orders.printSchema()


## Guided example and output explanation

The result has one row per `status`: `count` counts non-null order IDs and `sum` adds valid numeric values. The formatted plan reveals scans, exchanges, and aggregate operators; execution is only demanded by `show()`.

**Common mistake:** treating a Python variable as local data after a transformation. It is a distributed plan until an action produces a result. **Performance:** select/filter early and use native functions so Spark can optimize work.


In [ ]:
# Native expressions form one optimized plan; no large data is collected to Python.
typed_orders = (orders
    .withColumn("amount_num", F.col("amount").cast("double"))
    .withColumn("order_time", F.to_timestamp("order_ts"))
    .filter(F.col("amount_num") > 0))

summary = typed_orders.groupBy("status").agg(
    F.count("order_id").alias("orders"),
    F.round(F.sum("amount_num"), 2).alias("revenue"),
).orderBy(F.desc("revenue"))
summary.show()
summary.explain("formatted")


## Interactive experiment

Find revenue per country and explain which columns must cross the network in a distributed cluster.

Change one parameter, run the cell, and inspect both output and plan. Record your prediction before running it.


In [ ]:
# Experiment: partition count is observable, while an action materializes the plan.
print("Input partitions:", orders.rdd.getNumPartitions())
rebalanced = typed_orders.repartition(4, "customer_id")
print("Rebalanced partitions:", rebalanced.rdd.getNumPartitions())
print("Valid rows (action):", rebalanced.count())


## Real-world lens

An e-commerce team commonly applies this pattern to raw order data before deriving a customer metric table. In production, inputs have explicit schemas, rejected records are observable, output is Parquet, and the transformation is covered by tests—see `src/pyspark_learning`.

## Practice exercises

1. Filter for completed orders, select `customer_id` and a numeric amount, and predict the schema before calling `printSchema()`.
2. Calculate revenue per customer, sort descending, and explain whether grouping can shuffle.
3. Join the customer lookup, preserving all valid orders; verify the row count and investigate any increase.

### Debugging challenge

This fails because `amount` is a string: `orders.filter(F.col('amount') > 10)`. Repair it without a Python loop and decide what to do with malformed values.

### Concept check

Which of `filter`, `withColumn`, `groupBy`, `show`, and `count` are actions? Which may create a shuffle, and why?


## Recap and readiness checklist

- I can state the business and execution purpose of this module's operation.
- I can predict whether my code remains lazy or requests a result.
- I can inspect schema, sample safely, and use `explain('formatted')`.
- I can name at least one correctness risk and one performance risk.

Solutions are in the matching file under `solutions/`. Stop the session when working outside Jupyter with `spark.stop()`.
